In [1]:
# Keep imports and data/output paths relative to the repository root.
import os
import sys
from pathlib import Path
from datetime import datetime
from uuid import uuid4

for repository_dir in (Path("."), Path(".."), Path("../.."), Path("../../..")):
    if (repository_dir / "src" / "use_croma.py").is_file():
        os.chdir(repository_dir)
        break
else:
    raise FileNotFoundError("Open this notebook from inside the repository.")
if "src" not in sys.path:
    sys.path.insert(0, "src")

EXPERIMENT_NAME = "croma_sen1floods11/UperNet_head_S1S2"
NOTEBOOK_PATH = "experiments/croma_sen1floods11/UperNet_head_S1S2/01C_croma_on_sen1floods11_training_UperNet_S1S2.ipynb"


#### **Imports**

In [2]:
import torch
import torch.nn as nn
import torchvision.transforms.functional as FT
import torch.nn.functional as F
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import csv
import random
import json
import pandas as pd
from IPython.display import display

from PIL import Image
from pathlib import Path
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models.segmentation.fcn import FCNHead
from use_croma import PretrainedCROMA

In [3]:
from croma_upernet_pangaea import (
    UPerNetHead, filter_labeled_training_pairs, random_labeled_crop, PANGAEA_REVISION
)
from einops import rearrange
from segmentation_metrics import (binary_metrics, confusion_counts, METRIC_KEYS,
    metric_table, save_per_chip_metrics, clean_json, plot_metric_history)

CROP_SIZE = 120
IGNORE_INDEX = 255
SCHEMA = "croma_s1s2_upernet_pangaea_v1"
S2_BANDS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13]  # Exclude B10/cirrus.
# Same pinned PANGAEA dataset statistics as S1, selecting the optical channels.
PREPROCESSING = {
    "name": "pangaea_sen1floods11_optical_mean_std", "modality": "optical",
    "bands": S2_BANDS,
    "band_names": ["B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B9", "B11", "B12"],
    "mean": [1626.91600224, 1396.03470631, 1364.06118417, 1218.22847919,
             1466.07290663, 2386.90297537, 2845.61256277, 2622.95796892,
             3077.48221481, 486.87436782, 2030.64763024, 1179.16607221],
    "std": [700.17133846, 739.09452682, 735.2482388, 864.936695,
            776.8803358, 921.36834309, 1084.37346097, 1022.63418007,
            1196.44255318, 336.61105431, 980.87061347, 764.60836557],
    "nan_to_num": "numpy_default_before_normalization", "clip": None,
    "ignore_index": IGNORE_INDEX, "source_revision": PANGAEA_REVISION,
}

# Concatenate only for synchronized cropping/tiling; the model splits the sensors before encoding.
OPTICAL_PREPROCESSING = PREPROCESSING
PREPROCESSING = {
    "name": "pangaea_sen1floods11_joint_mean_std", "modality": "both",
    "channel_order": ["VV", "VH"] + OPTICAL_PREPROCESSING["band_names"],
    "mean": [-10.184408, -16.895273] + OPTICAL_PREPROCESSING["mean"],
    "std": [4.255339, 5.290568] + OPTICAL_PREPROCESSING["std"],
    "optical_bands": S2_BANDS, "nan_to_num": "numpy_default_before_normalization",
    "clip": None, "ignore_index": IGNORE_INDEX, "source_revision": PANGAEA_REVISION,
}

def sar_path(optical_path):
    return optical_path.parent.parent / "S1Hand" / optical_path.name.replace("_S2Hand.tif", "_S1Hand.tif")

def load_split_paths(csv_file, x_dir, y_dir):
    pairs = []
    with open(csv_file, newline="") as handle:
        for x_name, y_name in csv.reader(handle):
            chip = x_name.strip().removesuffix("_S1Hand.tif")
            if y_name.strip() != chip + "_LabelHand.tif":
                raise ValueError(f"Mismatched split row: {x_name}, {y_name}")
            x_path, y_path = Path(x_dir) / (chip + "_S2Hand.tif"), Path(y_dir) / y_name.strip()
            if not x_path.is_file() or not y_path.is_file():
                raise FileNotFoundError(f"Missing pair: {x_path}, {y_path}")
            with rasterio.open(x_path) as x, rasterio.open(y_path) as y:
                if (x.count != 13 or x.shape != y.shape or x.crs != y.crs
                        or not x.transform.almost_equals(y.transform, precision=1e-10)):
                    raise ValueError(f"Band/grid mismatch: {chip}")
            with rasterio.open(sar_path(x_path)) as sar, rasterio.open(y_path) as y:
                if (sar.count != 2 or sar.shape != y.shape or sar.crs != y.crs
                        or not sar.transform.almost_equals(y.transform, precision=1e-10)):
                    raise ValueError(f"SAR/optical/label grid mismatch: {chip}")
            pairs.append((x_path, y_path))
    return pairs

def load_image_pair(pair):
    with rasterio.open(pair[0]) as src:
        image = src.read(S2_BANDS, out_dtype="float32", masked=True).filled(np.nan)
    with rasterio.open(sar_path(pair[0])) as src:
        sar = src.read([1, 2], out_dtype="float32", masked=True).filled(np.nan)
    image = np.concatenate([sar, image], axis=0)
    with rasterio.open(pair[1]) as src:
        mask = src.read(1).astype(np.int64)
    return image, mask

def prepare_sample(data):
    raw, mask = data
    image = torch.as_tensor(np.nan_to_num(raw), dtype=torch.float32)
    mask = torch.as_tensor(mask, dtype=torch.long).clone()
    mask[mask == -1] = IGNORE_INDEX
    if image.ndim != 3 or image.shape[0] != 14 or image.shape[-2:] != mask.shape:
        raise ValueError("Expected aligned 14-channel SAR/optical image and label.")
    if not set(torch.unique(mask).tolist()).issubset({0, 1, IGNORE_INDEX}):
        raise ValueError("Unexpected label values.")
    mean = image.new_tensor(PREPROCESSING["mean"])[:, None, None]
    std = image.new_tensor(PREPROCESSING["std"])[:, None, None]
    image = (image - mean) / std
    if not torch.isfinite(image).all():
        raise ValueError("Nonfinite normalized multimodal input.")
    return image, mask

class Sen1Floods11Dataset(Dataset):
    def __init__(self, data_list, preprocess_func):
        self.data_list, self.preprocess_func = data_list, preprocess_func
    def __len__(self):
        return len(self.data_list)
    def __getitem__(self, index):
        return self.preprocess_func(load_image_pair(self.data_list[index]))

def audit_joint_pairs(pairs):
    rows = []
    for pair in pairs:
        raw, _ = load_image_pair(pair)
        valid = np.isfinite(raw).all(axis=0)
        rows.append({"image": str(pair[0]), "usable_pixel_fraction": float(valid.mean()),
                     "exclude_missing_joint": not bool(valid.any())})
    return pd.DataFrame(rows)

class CROMAUPerNet(nn.Module):
    def __init__(self, croma, head):
        super().__init__()
        self.croma = croma.requires_grad_(False)
        self.head = head
        self.croma.eval()
        if max(head.out_indices) >= len(croma.cross_encoder.layers):
            raise ValueError('Requested blocks exceed the joint encoder depth.')
        if head.config['in_channels'] != croma.encoder_dim:
            raise ValueError('Decoder channel count does not match CROMA.')

    def train(self, mode=True):
        super().train(mode)
        self.croma.eval()
        return self

    @torch.no_grad()
    def extract_features(self, images):
        p = self.croma.s1_encoder.patch_size
        height, width = images.shape[-2:]
        if images.shape[1] != 14 or height != width or height % p or (height // p)**2 != self.croma.num_patches:
            raise ValueError('Expected paired 14-channel tiles matching CROMA image_resolution.')
        bias = self.croma.attn_bias.to(device=images.device, dtype=images.dtype)
        # Full unimodal streams supply the pretrained joint cross-attention encoder.
        x = self.croma.s1_encoder(images[:, :2], bias)
        context = self.croma.s2_encoder(images[:, 2:], bias)
        features = []
        for index, (self_attention, cross_attention, ffn) in enumerate(self.croma.cross_encoder.layers):
            x = x + self_attention(x, bias)
            x = x + cross_attention(x, context, bias)
            x = x + ffn(x)
            if index in self.head.out_indices:
                selected = self.croma.cross_encoder.norm_out(x) if index == self.head.out_indices[-1] else x
                features.append(selected.transpose(1, 2).reshape(x.shape[0], x.shape[2], height // p, width // p))
            if index == self.head.out_indices[-1]:
                break
        return features

    def forward(self, images):
        logits = self.head(self.extract_features(images))
        return F.interpolate(logits, size=images.shape[-2:], mode='bilinear', align_corners=False)

def latest_completed_checkpoint(run_root):
    candidates = []
    for manifest in Path(run_root).glob('train_*/run.json'):
        metadata = json.loads(manifest.read_text(encoding='utf-8'))
        if metadata.get('schema') != SCHEMA:
            continue
        history_path = manifest.parent / 'croma_training_history.json'
        if not history_path.is_file():
            continue
        history = json.loads(history_path.read_text(encoding='utf-8'))
        checkpoint = manifest.parent / Path(metadata['checkpoint']).name
        if history.get('epochs_completed') == metadata.get('epochs') and checkpoint.is_file():
            candidates.append((metadata['started_at'], checkpoint))
    if not candidates:
        raise FileNotFoundError('No completed UPerNet run found. Run the training notebook first.')
    return max(candidates, key=lambda item: item[0])[1]


#### **First things first**


In [4]:
SEED = 234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

pretrained_path = "pretrained/CROMA_large.pt"

path_to_the_dataset = "dataset/sen1floods11_hand_labeled_only/HandLabeled/"

path_to_the_dataset_index = "dataset/sen1floods11_hand_labeled_only/flood_handlabeled/"

path_to_X = path_to_the_dataset + "S2Hand/"

path_to_Y = path_to_the_dataset + "LabelHand/"

path_to_train_csv = path_to_the_dataset_index + "flood_train_data.csv"

path_to_val_csv = path_to_the_dataset_index + "flood_valid_data.csv"

path_to_test_csv = path_to_the_dataset_index + "flood_test_data.csv"
path_to_test_bolivia_csv = path_to_the_dataset_index + "flood_bolivia_data.csv"

print(device)

cuda


In [5]:
CROP_SIZE = 120
IGNORE_INDEX = 255  # Equivalent to PANGAEA's -1 after remapping the same ignored pixels.
BATCH_SIZE = 8  # Real batch of 8: accumulation does not reproduce BatchNorm statistics.
TILE_BATCH_SIZE = 8

#### **Functions**

In [6]:
def preprocess_train(data):
    # Reject crops without supervision; valid land-only crops remain eligible.
    return prepare_sample(random_labeled_crop(*data, crop_size=CROP_SIZE))


def preprocess_eval(data):
    return prepare_sample(data)


#### Creating datasets

In [ ]:
# Verify official split membership, counts, disjointness, and S1/S2/label alignment.
split_files = {"train": "flood_train_data.csv", "valid": "flood_valid_data.csv",
               "non_bol": "flood_test_data.csv", "bol": "flood_bolivia_data.csv"}

expected_counts = {"train": 252, "valid": 89, "non_bol": 90, "bol": 15}

all_splits = {name: load_split_paths(Path(path_to_the_dataset_index) / filename, path_to_X, path_to_Y)
              for name, filename in split_files.items()}

for name, pairs in all_splits.items():
    assert len(pairs) == expected_counts[name], (name, len(pairs))
    assert len({x.stem for x, _ in pairs}) == len(pairs), f"Duplicates in {name}"
    
for i, name in enumerate(all_splits):
    for other in list(all_splits)[i+1:]:
        assert not ({x.stem for x, _ in all_splits[name]} & {x.stem for x, _ in all_splits[other]})
        
print("Verified S1/S2/label grids and disjoint split counts:", {k: len(v) for k, v in all_splits.items()})

train_data = all_splits["train"]
val_data = all_splits["valid"]

# Filter only the in-memory training list; preserve evaluation membership and source files.
train_input_audit = audit_joint_pairs(train_data)
val_input_audit = audit_joint_pairs(val_data)

excluded_missing_joint_train = [pair for pair, bad in zip(train_data, train_input_audit.exclude_missing_joint) if bad]
train_data = [pair for pair, bad in zip(train_data, train_input_audit.exclude_missing_joint) if not bad]

print(f"Excluded training images without usable paired S1/S2: {len(excluded_missing_joint_train)}")

display(train_input_audit[train_input_audit.exclude_missing_joint])
train_data, excluded_unlabeled_train = filter_labeled_training_pairs(train_data)

print(f"Excluded fully ignored training images: {len(excluded_unlabeled_train)}")


Verified S1/S2/label grids and disjoint split counts: {'train': 252, 'valid': 89, 'non_bol': 90, 'bol': 15}
Excluded training images without usable paired S1/S2: 0


,image,usable_pixel_fraction,exclude_missing_joint


Excluded fully ignored training images: 1


In [8]:
print(f"Train samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

Train samples: 251
Validation samples: 89


In [9]:
print(train_data[0])

(WindowsPath('dataset/sen1floods11_hand_labeled_only/HandLabeled/S2Hand/Ghana_103272_S2Hand.tif'), WindowsPath('dataset/sen1floods11_hand_labeled_only/HandLabeled/LabelHand/Ghana_103272_LabelHand.tif'))


In [10]:
train_dataset = Sen1Floods11Dataset(
    data_list=train_data,
    preprocess_func=preprocess_train
)

valid_dataset = Sen1Floods11Dataset(
    data_list=val_data,
    preprocess_func=preprocess_eval
)

#### Creating DataLoaders

In [11]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=torch.utils.data.DistributedSampler(train_dataset, num_replicas=1, rank=0),
    generator=torch.Generator().manual_seed(SEED),
    drop_last=True,
    num_workers=0,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# Training batch checks

images, masks = next(iter(train_loader))

print("Training images:", images.shape)
print("Training masks:", masks.shape)

print("Image min:", images.min().item())
print("Image max:", images.max().item())

print("Water pixels:", (masks == 1).sum().item())
print("Non-water pixels:", (masks == 0).sum().item())
print("Ignored pixels:", (masks == IGNORE_INDEX).sum().item())

assert torch.isfinite(images).all()

assert set(torch.unique(masks).tolist()).issubset(
    {0, 1, IGNORE_INDEX}
)

# Validation batch checks

val_images, val_masks = next(iter(valid_loader))

print("-" * 40)
print("Validation images:", val_images.shape)
print("Validation masks:", val_masks.shape)

Training images: torch.Size([8, 14, 120, 120])
Training masks: torch.Size([8, 120, 120])
Image min: -6.036761283874512
Image max: 5.543222427368164
Water pixels: 5494
Non-water pixels: 104995
Ignored pixels: 4711
----------------------------------------
Validation images: torch.Size([1, 14, 512, 512])
Validation masks: torch.Size([1, 512, 512])


##### Loading CROMA

In [12]:
croma = PretrainedCROMA(pretrained_path= pretrained_path, 
                        size='large', 
                        modality='both', 
                        image_resolution=120)

croma = croma.to(device)

Initializing SAR encoder
Initializing optical encoder
Initializing joint SAR-optical encoder


In [13]:
croma.requires_grad_(False) # Freeze the weights of CROMA
croma.eval()

PretrainedCROMA(
  (s1_encoder): ViT(
    (linear_input): Linear(in_features=128, out_features=1024, bias=True)
    (transformer): BaseTransformer(
      (layers): ModuleList(
        (0-11): 12 x ModuleList(
          (0): Attention(
            (to_qkv): Linear(in_features=1024, out_features=3072, bias=False)
            (to_out): Linear(in_features=1024, out_features=1024, bias=True)
            (input_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (1): FFN(
            (net): Sequential(
              (0): Linear(in_features=1024, out_features=4096, bias=True)
              (1): GELU(approximate='none')
              (2): Dropout(p=0.0, inplace=False)
              (3): Linear(in_features=4096, out_features=1024, bias=True)
            )
            (input_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
          )
        )
      )
      (norm_out): LayerNo

### Creating the head

In [14]:
# PANGAEA croma_joint.yaml: joint cross-attention outputs, zero-based indices.
# https://github.com/VMarsocci/pangaea-bench/blob/ca19b837a16797cd809225225b671473c7c93768/configs/encoder/croma_joint.yaml
OUT_INDICES = (3, 5, 7, 11)


In [15]:
# Full 1024-channel feature pyramid; 512-channel UPerNet decoder.
# BatchNorm2d is the single-process equivalent of upstream SyncBatchNorm.
HEAD_CONFIG = dict(in_channels=1024, channels=512, num_classes=2,
                   out_indices=list(OUT_INDICES), pool_scales=[1, 2, 3, 6])


In [ ]:
head = UPerNetHead(**HEAD_CONFIG).to(device)

In [ ]:
model = CROMAUPerNet(croma, head).to(device)

In [ ]:
EPOCHS = 80
EVAL_INTERVAL = 2

LR = 1e-4

WEIGHT_DECAY = 0.05

criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

optimizer = torch.optim.AdamW(model.head.parameters(), lr=LR,
                              betas=(0.9, 0.999), weight_decay=WEIGHT_DECAY)

# PANGAEA steps the scheduler after every optimizer update.
TOTAL_STEPS = len(train_loader) * EPOCHS

LR_MILESTONES = [int(TOTAL_STEPS * fraction) for fraction in (0.6, 0.9)]
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=LR_MILESTONES, gamma=0.1
)

In [ ]:
def train_one_epoch(model, train_loader, optimizer, criterion, device):
    
    model.train()
    total_loss = 0.0
    total_valid_pixels = 0
    
    counts = np.zeros(4, dtype=np.int64)
    
    for images, masks in train_loader:
        
        images = images.to(device)
        masks = masks.to(device)
        
        valid_pixels = (masks != IGNORE_INDEX).sum().item()
        
        if not valid_pixels:
            raise RuntimeError("Entire batch is unlabeled; inspect sampling before continuing.")
        
        optimizer.zero_grad(set_to_none=True)
        
        logits = model(images)
        
        loss = criterion(logits, masks)
        
        counts += confusion_counts(logits.detach().argmax(1).cpu().numpy(), masks.cpu().numpy())
        
        if not torch.isfinite(loss):
            raise FloatingPointError("Nonfinite training loss.")
        
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item() * valid_pixels
        total_valid_pixels += valid_pixels
        
    if not total_valid_pixels:
        raise RuntimeError("No valid training pixels.")
    
    # These metrics describe sampled training crops, not full-image training evaluation.
    train_one_epoch.last_metrics = binary_metrics(*counts)
    return total_loss / total_valid_pixels


In [20]:
@torch.inference_mode()
def sliding_window_logits(
    model,
    image,
    device,
    tile_size=120,
    stride=None,
    tile_batch_size=TILE_BATCH_SIZE
):
    """
    Predict segmentation logits for one complete image.

    Parameters
    ----------
    model:
        Frozen CROMA + trained UPerNet head.

    image:
        Tensor [14, H, W], already normalized.

    device:
        CUDA or CPU.

    tile_size:
        Spatial size of each CROMA input (120).

    stride:
        None uses the PANGAEA evenly spaced sliding-window grid.

    tile_batch_size:
        Number of windows processed simultaneously.

    Returns
    -------
    logits:
        Tensor [2, H, W] on CPU.
    """

    model.eval()

    C, H, W = image.shape

    assert C == 14
    assert H >= tile_size and W >= tile_size
    assert stride is None or 0 < stride <= tile_size

    # Generate starting positions along one dimension
    def get_starts(length):
        import math
        # Upstream default: 5 evenly spaced windows per axis for 512px / 120px.
        count = math.ceil(length / tile_size) if stride is None else math.ceil((length - tile_size) / stride) + 1
        return torch.linspace(0, length - tile_size, count).round().long().tolist()

    # All window coordinates
    positions = [
        (top, left)
        for top in get_starts(H)
        for left in get_starts(W)
    ]

    # Accumulate predictions and coverage counts
    logits_sum = torch.zeros((2, H, W), dtype=torch.float32)
    counts = torch.zeros((H, W), dtype=torch.float32)

    # Process windows in batches
    for start in range(0, len(positions), tile_batch_size):

        batch_positions = positions[start:start + tile_batch_size]

        # Extract tiles
        tiles = torch.stack([
            image[:, top:top + tile_size, left:left + tile_size]
            for top, left in batch_positions
        ])

        tiles = tiles.to(device)

        # Run frozen CROMA + UPerNet head
        outputs = model(tiles).cpu()

        # Place tile predictions back into the full image
        for (top, left), logits in zip(batch_positions, outputs):

            logits_sum[
                :, top:top + tile_size, left:left + tile_size
            ] += logits

            counts[
                top:top + tile_size, left:left + tile_size
            ] += 1

    assert (counts > 0).all(), "Some pixels were not covered."

    # Average logits wherever windows overlap
    return logits_sum / counts.unsqueeze(0)


import torch.nn.functional as F

@torch.inference_mode()
def validate(model, valid_loader, criterion, device):

    model.eval()

    total_loss = 0.0
    total_valid_pixels = 0

    TP = FP = FN = TN = 0

    for images, masks in valid_loader:

        # Process every complete image in the batch.
        for image, mask in zip(images, masks):
            mask = mask.cpu()
            
            
            valid = mask != IGNORE_INDEX

            if not valid.any():
                continue

            # Reconstruct logits using overlapping windows.
            logits = sliding_window_logits(
                model,
                image,
                device,
                tile_size=CROP_SIZE,
                stride=None,
                tile_batch_size=TILE_BATCH_SIZE
            )
            
            logits_device = logits.unsqueeze(0).to(device)
            mask_device = mask.unsqueeze(0).to(device=device, dtype=torch.long)

            loss = criterion(logits_device, mask_device)

            image_valid_pixels = valid.sum().item()

            total_loss += loss.item() * image_valid_pixels
            total_valid_pixels += image_valid_pixels

            prediction = logits.argmax(dim=0)

            TP += ((prediction == 1) & (mask == 1) & valid).sum().item()
            FP += ((prediction == 1) & (mask == 0) & valid).sum().item()
            FN += ((prediction == 0) & (mask == 1) & valid).sum().item()
            TN += ((prediction == 0) & (mask == 0) & valid).sum().item()

    if total_valid_pixels == 0:
        raise RuntimeError("No valid validation pixels.")

    val_loss = total_loss / total_valid_pixels

    return {"val_loss": val_loss, **binary_metrics(TP, FP, FN, TN)}


In [21]:
metrics = validate(model, valid_loader, criterion, device)
display(metric_table({"validation_baseline": metrics}).T)

split,validation_baseline
val_loss,6.539763e-01
accuracy,8.854651e-01
water_iou,3.758608e-04
nonwater_iou,8.854601e-01
miou,4.429180e-01
precision,9.864783e-03
recall,3.905962e-04
f1,7.514391e-04
omission_error,9.996094e-01
commission_error,9.901352e-01


In [ ]:
model_name = "croma_large_upernet_pangaea_s1s2.pt"
run_id = datetime.now().astimezone().strftime("%Y-%m-%d_%H-%M-%S_%f") + "_" + uuid4().hex[:8]
checkpoint_dir = Path("runs") / EXPERIMENT_NAME / f"train_{Path(model_name).stem}_{run_id}"
checkpoint_dir.mkdir(parents=True, exist_ok=False)
checkpoint_path = checkpoint_dir / model_name
history_path = checkpoint_dir / "croma_training_history.json"

print(f"Run directory: {checkpoint_dir}")
train_input_audit.to_csv(checkpoint_dir / "training_input_quality.csv", index=False)
val_input_audit.to_csv(checkpoint_dir / "validation_input_quality.csv", index=False)

run_config = {
    "experiment": EXPERIMENT_NAME, "model_name": checkpoint_path.stem,
    "phase": "train", "run_id": run_id,
    "started_at": datetime.now().astimezone().isoformat(),
    "notebook": NOTEBOOK_PATH, "schema": SCHEMA,
    "pangaea_revision": PANGAEA_REVISION, "modality": "both",
    "pretrained_weights": pretrained_path, "head_config": HEAD_CONFIG,
    "preprocessing": PREPROCESSING, "crop_size": CROP_SIZE,
    "split_counts": {k: len(v) for k, v in all_splits.items()},
    "effective_training_samples": len(train_data),
    "epochs": EPOCHS, "eval_interval": EVAL_INTERVAL, "seed": SEED,
    "batch_size": BATCH_SIZE, "drop_last": True, "precision": "fp32",
    "backbone_frozen": True, "loss": "unweighted_cross_entropy",
    "ignore_index": IGNORE_INDEX, "optimizer": "AdamW",
    "learning_rate": LR, "weight_decay": WEIGHT_DECAY,
    "scheduler": {"name": "MultiStepLR", "milestones": LR_MILESTONES,
                  "gamma": 0.1, "step_unit": "optimizer_update"},
    "checkpoint_selection": "validation_miou",
    "training_sampling": {"exclude_fully_ignored_images": True,
                          "reject_fully_ignored_crops": True,
                          "max_crop_attempts": 32,
                          "fallback": "random_crop_containing_random_valid_pixel",
                          "water_oversampling": False,
                          "exclude_images_without_usable_joint": True},
    "excluded_missing_joint_training_images": [str(pair[0]) for pair in excluded_missing_joint_train],
    "excluded_unlabeled_training_images": [str(pair[0]) for pair in excluded_unlabeled_train],
    "inference_grid": "linspace_ceil_image_div_tile",
    "runtime_adaptations": ["single_process_BatchNorm2d", "num_workers_0", "ignore_label_255"],
    "checkpoint": checkpoint_path.as_posix(), "history": history_path.as_posix(),
}

with (checkpoint_dir / "run.json").open("w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=4)

# Include the untrained validation baseline, as PANGAEA evaluates at epoch zero.
best_val_miou = metrics["miou"]
best_val_iou = metrics["water_iou"]
best_epoch = 0

def save_best(epoch, current_metrics):
    torch.save({
        "schema": SCHEMA, "pangaea_revision": PANGAEA_REVISION,
        "head_config": HEAD_CONFIG, "preprocessing": PREPROCESSING,
        "crop_size": CROP_SIZE, "pretrained_weights": pretrained_path,
        "epoch": epoch, "head_state_dict": model.head.state_dict(),
        "best_val_miou": current_metrics["miou"],
        "best_val_water_iou": current_metrics["water_iou"],
        "val_miou": current_metrics["miou"], "val_loss": current_metrics["val_loss"],
        "seed": SEED, "run_config": run_config,
        "validation_metrics": clean_json(current_metrics),
    }, checkpoint_path)

save_best(0, metrics)
history = {key: [] for key in ("train_loss", "val_loss", "val_water_iou", "val_miou", "learning_rate")}
baseline_metrics = dict(metrics)
for key in METRIC_KEYS:
    history.setdefault("val_" + key, [])
    history.setdefault("train_" + key, [])
validation_rows = [{"epoch": 0, **metrics}]
pd.DataFrame(validation_rows).to_csv(checkpoint_dir / "validation_metrics.csv", index=False)


for epoch in range(EPOCHS):

    train_loader.sampler.set_epoch(epoch)
    epoch_lr = optimizer.param_groups[0]["lr"]

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)


    evaluate_now = (epoch + 1) % EVAL_INTERVAL == 0 or epoch + 1 == EPOCHS
    metrics = validate(model, valid_loader, criterion, device) if evaluate_now else None


    history["train_loss"].append(train_loss)
    train_metrics = getattr(train_one_epoch, "last_metrics", {})

    for key in METRIC_KEYS:
        history["train_" + key].append(train_metrics.get(key))
        if key not in ("water_iou", "miou"):
            history["val_" + key].append(metrics.get(key) if metrics is not None else None)

    if metrics is not None:
        validation_rows.append({"epoch": epoch + 1, **metrics})
        pd.DataFrame(validation_rows).to_csv(checkpoint_dir / "validation_metrics.csv", index=False)
        display(metric_table({f"validation_epoch_{epoch+1}": metrics}).T)

    history["learning_rate"].append(epoch_lr)

    for key, metric in (("val_loss", "val_loss"), ("val_water_iou", "water_iou"), ("val_miou", "miou")):
        history[key].append(metrics[metric] if metrics is not None else None)

    if metrics is not None and metrics["miou"] > best_val_miou:
        best_val_miou = metrics["miou"]
        best_val_iou = metrics["water_iou"]
        best_epoch = epoch + 1
        save_best(best_epoch, metrics)

    with history_path.open("w", encoding="utf-8") as f:
        json.dump(clean_json({"epochs_completed": epoch + 1, "best_epoch": best_epoch,
                   "best_val_miou": best_val_miou, "best_val_water_iou": best_val_iou,
                   "baseline_metrics": baseline_metrics, "history": history}), f, indent=4, allow_nan=False)
    message = f"Epoch [{epoch+1:02d}/{EPOCHS}] | LR: {epoch_lr:.2e} | Train Loss: {train_loss:.4f}"

    if metrics is not None:
        message += (f" | Val Loss: {metrics['val_loss']:.4f}"
                    f" | Water IoU: {metrics['water_iou']:.4f} | mIoU: {metrics['miou']:.4f}")
    pd.DataFrame(history).rename_axis("epoch_index").to_csv(checkpoint_dir / "all_metrics_history.csv")
    print(message)

print(f"Best checkpoint: epoch {best_epoch}, mIoU {best_val_miou:.4f}, water IoU {best_val_iou:.4f}")


Run directory: runs\croma_sen1floods11\UperNet_head_S1S2\train_croma_large_upernet_pangaea_s1s2_2026-10-26_10-57-52_909543_d8872ab6
Epoch [01/80] | LR: 1.00e-04 | Train Loss: 0.3820


In [ ]:
print(
    "CROMA trainable parameters:",
    sum(p.numel() for p in model.croma.parameters()
        if p.requires_grad)
)

print(
    "Head trainable parameters:",
    sum(p.numel() for p in model.head.parameters()
        if p.requires_grad)
)

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
eval_epochs = [i + 1 for i, value in enumerate(history["val_loss"]) if value is not None]

plt.figure(figsize=(10, 5))

plt.plot(
    epochs,
    history["train_loss"],
    label="Training loss",
    linewidth=2
)

plt.plot(
    eval_epochs,
    [value for value in history["val_loss"] if value is not None],
    label="Validation loss",
    linewidth=2
)

plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("CROMA Large + UPerNet — Training and Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(checkpoint_dir / "training_loss.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    eval_epochs,
    [value for value in history["val_water_iou"] if value is not None],
    label="Water IoU",
    linewidth=2
)

plt.plot(
    eval_epochs,
    [value for value in history["val_miou"] if value is not None],
    label="Binary mIoU",
    linewidth=2
)

plt.axvline(
    best_epoch,
    linestyle="--",
    alpha=0.6,
    label=f"Best checkpoint (epoch {best_epoch})"
)

plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.title("CROMA Large + UPerNet — Validation Segmentation Performance")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(checkpoint_dir / "validation_iou.png", dpi=150, bbox_inches="tight")
plt.show()
plot_metric_history(history, checkpoint_dir)
plt.show()
